Ali Issabayev is responsible for this code

### RT-DETR Training on TACO Supercategory Dataset

#### 1. Environment Setup

In [5]:
!pip install ultralytics>=8.3.0 -q
!pip install torch torchvision -q
!pip install pandas matplotlib seaborn -q

#### 2. Dataset Configuration

Assuming the TACO dataset has already been converted to YOLO format with supercategories.

In [6]:
from pathlib import Path

DATASET_ROOT = Path("/home/default/Desktop/project/taco_yolo_supercat")
YAML_CONFIG = DATASET_ROOT / "taco.yaml"

# Verify dataset structure
print("Dataset structure:")
for split in ["train", "val", "test"]:
    img_dir = DATASET_ROOT / f"images/{split}"
    lbl_dir = DATASET_ROOT / f"labels/{split}"
    if img_dir.exists() and lbl_dir.exists():
        n_imgs = len(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")))
        n_lbls = len(list(lbl_dir.glob("*.txt")))
        print(f"  {split}: {n_imgs} images, {n_lbls} labels")
    else:
        print(f"  {split}: NOT FOUND")

# Load and display class names
if YAML_CONFIG.exists():
    import yaml
    with open(YAML_CONFIG, 'r') as f:
        config = yaml.safe_load(f)
    print(f"\nNumber of classes: {len(config['names'])}")
    print(f"Classes: {list(config['names'].values())[:10]}...")

Dataset structure:
  train: 570 images, 1050 labels
  val: 125 images, 225 labels
  test: 138 images, 225 labels

Number of classes: 28
Classes: ['aluminium foil', 'battery', 'blister pack', 'bottle', 'bottle cap', 'broken glass', 'can', 'carton', 'cigarette', 'cup']...


#### 3. Train RT-DETR

In [ ]:
from ultralytics import RTDETR
import os
os.environ["WANDB_DISABLED"] = "true"

model = RTDETR('rtdetr-l.pt')

# Training configuration
results = model.train(
    data=str(YAML_CONFIG),
    epochs=100,
    imgsz=640,
    batch=8,
    patience=30,
    device=0,
    project='runs/rtdetr',
    name='taco_simple',
    exist_ok=True,
    pretrained=True,
    
    # RT-DETR works better with AdamW
    optimizer='AdamW',
    lr0=0.0001,
    
    # Basic augmentations
    mosaic=0.5,
    copy_paste=0.3,   # Good for small objects
    mixup=0.1,
    
    # Enable validation
    val=True,
    save=True,
    plots=True,
)

print("\n" + "="*50)
print("Training completed!")
print(f"Best model: runs/rtdetr/taco_simple/weights/best.pt")
print("="*50)

New https://pypi.org/project/ultralytics/8.3.231 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.229 🚀 Python-3.10.18 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 15948MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/default/Desktop/project/taco_yolo_supercat/taco.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=0.5, multi_scale=False, name=taco_si

/home/default/.local/lib/python3.10/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:157.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      7.55G      1.532      1.516     0.7501          5        640: 100% ━━━━━━━━━━━━ 132/132 2.0it/s 1:070.4sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 4.3it/s 3.5s0.2s
                   all        225        823   6.04e-05     0.0233   0.000561   0.000197

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/100      7.55G      1.466     0.8144     0.5438         53        640: 0% ──────────── 0/132  0.6s

/home/default/.local/lib/python3.10/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:157.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100      7.55G      1.104      1.216     0.4337          9        640: 100% ━━━━━━━━━━━━ 132/132 2.1it/s 1:040.4sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 4.3it/s 3.5s0.2s
                   all        225        823    6.2e-05     0.0239   0.000473   0.000195

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/100      7.55G      1.054      1.121     0.3987         34        640: 0% ──────────── 0/132  0.6s

...

     91/100      7.71G     0.4079     0.5734    0.07999          5        640: 100% ━━━━━━━━━━━━ 132/132 2.0it/s 1:050.4sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 4.2it/s 3.6s0.3s
                   all        225        823      0.408      0.167      0.162      0.131

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     92/100      7.71G     0.5166     0.4012    0.09205         13        640: 0% ──────────── 0/132  0.5s

/home/default/.local/lib/python3.10/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:157.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     92/100      7.71G     0.4003       0.58    0.09022          5        640: 100% ━━━━━━━━━━━━ 132/132 2.1it/s 1:040.4sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 4.2it/s 3.6s0.3s
                   all        225        823      0.358      0.186      0.161       0.13

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     93/100      7.71G     0.3155     0.6497    0.06471         18        640: 0% ──────────── 0/132  0.5s

/home/default/.local/lib/python3.10/site-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:157.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     93/100      7.71G     0.4075     0.5683    0.08499          2        640: 100% ━━━━━━━━━━━━ 132/132 2.1it/s 1:040.4sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 4.2it/s 3.6s0.3s
                   all        225        823      0.399      0.176       0.16       0.13
EarlyStopping: Training stopped early as no improvement observed in last 30 epochs. Best results observed at epoch 63, best model saved as best.pt.
To update EarlyStopping(patience=30) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

93 epochs completed in 1.794 hours.
Optimizer stripped from /home/default/Desktop/project/object_detection/runs/rtdetr/taco_simple/weights/last.pt, 66.3MB
Optimizer stripped from /home/default/Desktop/project/object_detection/runs/rtdetr/taco_simple/weights/best.pt, 66.3MB

Validating /home/default/Desktop/project/object_detection/runs/rtdetr/taco_simple/weights/best.pt...
U

#### 4. Validation Results

In [ ]:
model = RTDETR('runs/rtdetr/taco_simple/weights/best.pt')

metrics = model.val(
    data=str(YAML_CONFIG), 
    imgsz=640,
    batch=8
)

print("\nOfficial Validation Metrics:")
print(f"  mAP@50: {metrics.box.map50:.4f}")
print(f"  mAP@50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")

if metrics.box.mp > 0 and metrics.box.mr > 0:
    f1 = 2 * (metrics.box.mp * metrics.box.mr) / (metrics.box.mp + metrics.box.mr)
    print(f"  F1 Score: {f1:.4f}")

Ultralytics 8.3.229 🚀 Python-3.10.18 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 15948MiB)
rt-detr-l summary: 302 layers, 32,041,280 parameters, 0 gradients, 103.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5938.8±741.4 MB/s, size: 1010.5 KB)
val: Scanning /home/default/Desktop/project/taco_yolo_supercat/labels/val.cache... 225 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 225/225 602.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 3.5it/s 8.4s0.3s
                   all        225        823      0.367      0.185       0.17       0.14
        aluminium foil          8          9        0.8      0.222      0.235      0.218
               battery          1          1          0          0          0          0
          blister pack          2          2          0          0          0          0
                bottle         51         68      0.546      0.544        0.5      0.391
            